<a href="https://colab.research.google.com/github/iinkc00l/university-public-library-dataset/blob/main/PublicLib_DIB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL GOOGLE COLAB PREPROCESSING SCRIPT
#
# OUTPUTS:
#   1. df_clean : cleaned transaction-level dataset
#   2. R0       : initial binary user-item interaction matrix
#
# CORE VARIABLES:
#   User ID : Membership Number
#   Item ID : BIBID
#
# INTERACTION:
#   - If Borrowed exists -> validate and use it
#   - If Borrowed does not exist -> create Borrowed = 1
#
# ADDITIONAL AUDIT:
#   - encoding detection
#   - malformed rows
#   - missing core values
#   - invalid Borrowed values
#   - exact duplicates
#   - repeated user-item interactions
#   - dropped records
#   - retained records
#   - unique users/items
#   - R0 dimensions
#   - density/sparsity
# ============================================================


# ============================================================
# 1. INSTALL AND IMPORT LIBRARIES
# ============================================================

!pip -q install pandas numpy scipy chardet openpyxl

import os
import gc
import chardet
import numpy as np
import pandas as pd

from scipy import sparse

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("✅ Libraries loaded.")


# ============================================================
# 2. DATASET PATH
# ============================================================

path = "/content/drive/MyDrive/Data/datasetPublicLib.csv"

assert os.path.exists(path), f"❌ File not found: {path}"

print("\n✅ Dataset found:")
print(path)


# ============================================================
# 3. HELPER FUNCTION FOR COLUMN IDENTIFICATION
# ============================================================

def pick_col(df, candidates):
    """
    Finds a column using exact or case-insensitive matching.
    """

    # Exact match
    for c in candidates:
        if c in df.columns:
            return c

    # Case-insensitive match
    lower_map = {
        str(c).lower(): c
        for c in df.columns
    }

    for c in candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]

    return None


# ============================================================
# 4. DETECT FILE ENCODING
# ============================================================

with open(path, "rb") as f:
    raw_sample = f.read(200_000)

encoding_result = chardet.detect(raw_sample)

enc_guess = encoding_result.get("encoding")
enc_confidence = encoding_result.get("confidence")

print("\n" + "=" * 70)
print("ENCODING DETECTION")
print("=" * 70)

print(f"Detected encoding : {enc_guess}")
print(f"Confidence        : {enc_confidence}")


# ============================================================
# 5. ROBUST CSV LOADING WITH BAD-LINE AUDIT
# ============================================================

candidate_encodings = []

for enc in [
    enc_guess,
    "utf-8-sig",
    "utf-8",
    "utf8",
    "latin1",
    "ISO-8859-1",
    "Windows-1252"
]:
    if enc and enc not in candidate_encodings:
        candidate_encodings.append(enc)


df_raw = None
successful_encoding = None
bad_lines_final = []


for enc in candidate_encodings:

    bad_lines_temp = []

    def capture_bad_line(fields):
        """
        Stores malformed CSV rows and skips them.
        """
        bad_lines_temp.append(fields)
        return None

    try:

        temp_df = pd.read_csv(
            path,
            encoding=enc,
            sep=None,
            engine="python",
            on_bad_lines=capture_bad_line
        )

        df_raw = temp_df
        successful_encoding = enc
        bad_lines_final = bad_lines_temp

        print(
            f"\n✅ Successfully read dataset "
            f"using encoding: {enc}"
        )

        break

    except Exception as e:

        print(f"⚠️ Failed with encoding {enc}")
        print(f"   Reason: {str(e)[:120]}")


if df_raw is None:
    raise RuntimeError(
        "❌ Dataset could not be read using available encodings."
    )


N_RAW_PARSED = len(df_raw)


print("\n" + "=" * 70)
print("RAW DATASET")
print("=" * 70)

print(f"Rows successfully parsed : {len(df_raw):,}")
print(f"Columns                  : {df_raw.shape[1]:,}")
print(f"Malformed rows skipped   : {len(bad_lines_final):,}")
print(f"Encoding used            : {successful_encoding}")


# ============================================================
# 6. IDENTIFY CORE RECOMMENDER COLUMNS
# ============================================================

user_col = pick_col(
    df_raw,
    [
        "Membership Number",
        "membership_number",
        "MembershipNumber"
    ]
)


# IMPORTANT:
# BIBID is used as ITEM identifier
item_col = pick_col(
    df_raw,
    [
        "BIBID",
        "bibid",
        "Bibid"
    ]
)


borrow_col = pick_col(
    df_raw,
    [
        "Borrowed",
        "borrowed",
        "rating",
        "Rating",
        "label",
        "Label"
    ]
)


# ============================================================
# 7. CREATE BORROWED COLUMN IF IT DOES NOT EXIST
# ============================================================

borrowed_column_created = False


if borrow_col is None:

    print("\n⚠️ No Borrowed column was found.")

    print(
        "Creating a new 'Borrowed' column "
        "with value 1 for every transaction."
    )

    df_raw["Borrowed"] = 1

    borrow_col = "Borrowed"

    borrowed_column_created = True


print("\n" + "=" * 70)
print("CORE COLUMN IDENTIFICATION")
print("=" * 70)

print(f"User column        : {user_col}")
print(f"Item column        : {item_col}")
print(f"Interaction column : {borrow_col}")

print(
    f"Borrowed created   : "
    f"{'YES' if borrowed_column_created else 'NO'}"
)


if user_col is None:
    raise ValueError(
        "❌ Membership Number could not be identified."
    )


if item_col is None:
    raise ValueError(
        "❌ BIBID could not be identified."
    )


# ============================================================
# 8. INITIAL MISSING-VALUE AUDIT
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE AUDIT — RAW DATA")
print("=" * 70)


raw_missing_user = (
    df_raw[user_col]
    .isna()
)


raw_missing_item = (
    df_raw[item_col]
    .isna()
)


raw_missing_borrow = (
    df_raw[borrow_col]
    .isna()
)


print(
    f"Missing {user_col:<25}: "
    f"{raw_missing_user.sum():>10,}"
)

print(
    f"Missing {item_col:<25}: "
    f"{raw_missing_item.sum():>10,}"
)

print(
    f"Missing {borrow_col:<25}: "
    f"{raw_missing_borrow.sum():>10,}"
)


if borrowed_column_created:

    print(
        "\nℹ️ Borrowed did not exist in the original dataset."
    )

    print(
        "   Borrowed = 1 was assigned "
        "to every parsed transaction row."
    )


# ============================================================
# 9. FULL MISSING-VALUE TABLE
# ============================================================

missing_table = pd.DataFrame({

    "Column":
        df_raw.columns,

    "Missing_Count":
        df_raw.isna().sum().values,

    "Missing_Percent":
        (
            df_raw.isna().sum().values
            /
            len(df_raw)
            *
            100
        )
})


missing_table = (
    missing_table
    .sort_values(
        "Missing_Count",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nMissing-value summary for all variables:")

display(missing_table)


# ============================================================
# 10. CREATE WORKING COPY
# ============================================================

df_work = df_raw.copy()


print(
    f"\nInitial working records: "
    f"{len(df_work):,}"
)


# ============================================================
# 11. STANDARDIZE USER AND ITEM IDENTIFIERS
# ============================================================

for c in [
    user_col,
    item_col
]:

    # Preserve true missing values
    df_work[c] = (
        df_work[c]
        .astype("string")
    )

    # Remove leading/trailing spaces
    df_work[c] = (
        df_work[c]
        .str.strip()
    )

    # Treat common missing strings as missing
    df_work[c] = (
        df_work[c]
        .replace({
            "": pd.NA,
            " ": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA
        })
    )


missing_user_mask = (
    df_work[user_col]
    .isna()
)


missing_item_mask = (
    df_work[item_col]
    .isna()
)


print("\n" + "=" * 70)
print("IDENTIFIER QUALITY AFTER STANDARDIZATION")
print("=" * 70)


print(
    f"Missing/blank Membership Number : "
    f"{missing_user_mask.sum():,}"
)

print(
    f"Missing/blank BIBID             : "
    f"{missing_item_mask.sum():,}"
)


# ============================================================
# 12. VALIDATE BORROWED VARIABLE
# ============================================================

# Preserve original values for auditing
df_work["_Borrowed_original"] = (
    df_work[borrow_col]
)


# Convert to numeric
borrow_numeric = pd.to_numeric(
    df_work[borrow_col],
    errors="coerce"
)


# ----------------------------
# Missing Borrowed
# ----------------------------

missing_borrow_mask = (
    df_work[borrow_col]
    .isna()
)


# ----------------------------
# Invalid non-numeric Borrowed
# ----------------------------

invalid_borrow_mask = (

    df_work[borrow_col]
    .notna()

    &

    borrow_numeric
    .isna()
)


# ----------------------------
# Negative Borrowed
# ----------------------------

negative_borrow_mask = (

    borrow_numeric
    .notna()

    &

    (
        borrow_numeric < 0
    )
)


print("\n" + "=" * 70)
print("BORROWED VARIABLE QUALITY")
print("=" * 70)


print(
    f"Missing Borrowed values     : "
    f"{missing_borrow_mask.sum():,}"
)

print(
    f"Invalid/non-numeric values  : "
    f"{invalid_borrow_mask.sum():,}"
)

print(
    f"Negative Borrowed values    : "
    f"{negative_borrow_mask.sum():,}"
)


if invalid_borrow_mask.sum() > 0:

    print(
        "\nExamples of invalid Borrowed values:"
    )

    display(
        df_work.loc[
            invalid_borrow_mask,
            [
                user_col,
                item_col,
                borrow_col
            ]
        ].head(20)
    )


# ============================================================
# 13. DEFINE RECORDS THAT CANNOT PROCEED
# ============================================================

# A record cannot proceed if:
#
# 1. Membership Number is missing
# 2. BIBID is missing
# 3. Borrowed is missing
# 4. Borrowed is non-numeric
# 5. Borrowed is negative


invalid_core_mask = (

    missing_user_mask

    |

    missing_item_mask

    |

    missing_borrow_mask

    |

    invalid_borrow_mask

    |

    negative_borrow_mask
)


rows_invalid_core = int(
    invalid_core_mask.sum()
)


print("\n" + "=" * 70)
print("CORE VALIDATION")
print("=" * 70)


print(
    f"Rows failing ≥1 mandatory rule : "
    f"{rows_invalid_core:,}"
)

print(
    f"Rows eligible to continue      : "
    f"{len(df_work) - rows_invalid_core:,}"
)


# Save invalid rows for auditing
df_dropped_core = (
    df_work
    .loc[invalid_core_mask]
    .copy()
)


# Remove invalid rows
df_work = (
    df_work
    .loc[~invalid_core_mask]
    .copy()
)


print(
    f"\n✅ Remaining rows after "
    f"core validation: {len(df_work):,}"
)


# ============================================================
# 14. CONVERT BORROWED INTO BINARY IMPLICIT FEEDBACK
# ============================================================

borrow_numeric_clean = pd.to_numeric(
    df_work[borrow_col],
    errors="raise"
)


# Count positive values >1 before conversion
nonbinary_positive_count = int(
    (
        borrow_numeric_clean > 1
    ).sum()
)


# Binary implicit-feedback conversion:
#
# Borrowed > 0 -> 1
# Borrowed = 0 -> 0

df_work[borrow_col] = (

    borrow_numeric_clean > 0

).astype(np.int8)


print("\n" + "=" * 70)
print("IMPLICIT FEEDBACK TRANSFORMATION")
print("=" * 70)


print(
    f"Values >1 converted to 1 : "
    f"{nonbinary_positive_count:,}"
)

print(
    f"Binary positive rows (1) : "
    f"{(df_work[borrow_col] == 1).sum():,}"
)

print(
    f"Binary zero rows (0)     : "
    f"{(df_work[borrow_col] == 0).sum():,}"
)


# ============================================================
# 15. EXACT DUPLICATE TRANSACTION AUDIT
# ============================================================

# Use original dataset columns for duplicate checking.
# Temporary audit columns are excluded.

duplicate_check_cols = [

    c

    for c in df_raw.columns

    if c in df_work.columns
]


exact_duplicate_mask = (

    df_work
    .duplicated(
        subset=duplicate_check_cols,
        keep="first"
    )
)


n_exact_duplicates = int(
    exact_duplicate_mask.sum()
)


print("\n" + "=" * 70)
print("EXACT DUPLICATE TRANSACTION AUDIT")
print("=" * 70)


print(
    f"Exact duplicate rows identified : "
    f"{n_exact_duplicates:,}"
)


# Save duplicate rows
df_dropped_duplicates = (

    df_work
    .loc[exact_duplicate_mask]
    .copy()
)


# Remove exact duplicates
df_work = (

    df_work
    .loc[~exact_duplicate_mask]
    .copy()
)


print(
    f"Rows after exact duplicate removal : "
    f"{len(df_work):,}"
)


# ============================================================
# 16. CREATE FINAL CLEANED TRANSACTION DATASET
# ============================================================

# Remove temporary audit variable
df_work = (
    df_work
    .drop(
        columns=[
            "_Borrowed_original"
        ],
        errors="ignore"
    )
)


df_clean = (
    df_work
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("OBJECT 1 — CLEANED TRANSACTION DATASET")
print("=" * 70)


print(
    f"Shape                     : "
    f"{df_clean.shape}"
)

print(
    f"Rows                      : "
    f"{len(df_clean):,}"
)

print(
    f"Columns                   : "
    f"{df_clean.shape[1]:,}"
)

print(
    f"Unique users              : "
    f"{df_clean[user_col].nunique():,}"
)

print(
    f"Unique BIBIDs             : "
    f"{df_clean[item_col].nunique():,}"
)

print(
    f"Positive transaction rows : "
    f"{(df_clean[borrow_col] == 1).sum():,}"
)

print(
    f"Zero transaction rows     : "
    f"{(df_clean[borrow_col] == 0).sum():,}"
)


# ============================================================
# 17. REPEATED USER–ITEM INTERACTION AUDIT
# ============================================================

# Repeated borrowing is NOT automatically removed.
#
# Example:
#
# User U001 borrows BIBID 1001 in 2022
# User U001 borrows BIBID 1001 again in 2024
#
# Both rows remain in df_clean.
#
# For binary R0:
#
# R0(U001, 1001) = 1


pair_counts = (

    df_clean

    .groupby(
        [
            user_col,
            item_col
        ],
        observed=True
    )

    .size()

    .reset_index(
        name="transaction_count"
    )
)


repeated_pairs = (

    pair_counts[
        pair_counts["transaction_count"] > 1
    ]

    .copy()
)


extra_repeated_transactions = int(

    (
        repeated_pairs[
            "transaction_count"
        ]
        -
        1
    ).sum()
)


print("\n" + "=" * 70)
print("REPEATED USER–ITEM INTERACTION AUDIT")
print("=" * 70)


print(
    f"Unique user-item pairs     : "
    f"{len(pair_counts):,}"
)

print(
    f"Pairs appearing >1 time    : "
    f"{len(repeated_pairs):,}"
)

print(
    f"Extra repeated transactions: "
    f"{extra_repeated_transactions:,}"
)


if len(repeated_pairs) > 0:

    print(
        "\nExamples of repeated "
        "user-item pairs:"
    )

    display(

        repeated_pairs

        .sort_values(
            "transaction_count",
            ascending=False
        )

        .head(20)
    )


# ============================================================
# 18. COLLAPSE TRANSACTIONS INTO UNIQUE USER–ITEM PAIRS
# ============================================================

# Multiple transactions for the same user-BIBID pair
# are collapsed using maximum Borrowed value.
#
# If any valid interaction is positive:
#
# R0(u,i) = 1


ui_binary = (

    df_clean

    .groupby(
        [
            user_col,
            item_col
        ],
        observed=True,
        as_index=False
    )[borrow_col]

    .max()
)


print("\n" + "=" * 70)
print("USER–ITEM PAIR COLLAPSING")
print("=" * 70)


print(
    f"Transaction rows in df_clean : "
    f"{len(df_clean):,}"
)

print(
    f"Unique user-item pairs       : "
    f"{len(ui_binary):,}"
)

print(
    f"Positive user-item pairs     : "
    f"{(ui_binary[borrow_col] == 1).sum():,}"
)


# ============================================================
# 19. CREATE USER AND ITEM INDEX MAPPINGS
# ============================================================

users = pd.Index(

    sorted(
        df_clean[
            user_col
        ].unique()
    ),

    name=user_col
)


items = pd.Index(

    sorted(
        df_clean[
            item_col
        ].unique()
    ),

    name=item_col
)


user_to_idx = {

    user: idx

    for idx, user

    in enumerate(users)
}


item_to_idx = {

    item: idx

    for idx, item

    in enumerate(items)
}


user_index_mapping = pd.DataFrame({

    "user_idx":
        range(len(users)),

    user_col:
        users
})


item_index_mapping = pd.DataFrame({

    "item_idx":
        range(len(items)),

    item_col:
        items
})


print("\n" + "=" * 70)
print("INDEX MAPPINGS")
print("=" * 70)


print(
    f"Unique users : "
    f"{len(users):,}"
)

print(
    f"Unique BIBIDs: "
    f"{len(items):,}"
)


# ============================================================
# 20. BUILD INITIAL R0 AS SPARSE MATRIX
# ============================================================

positive_pairs = (

    ui_binary[
        ui_binary[borrow_col] == 1
    ]

    .copy()
)


row_indices = (

    positive_pairs[user_col]

    .map(user_to_idx)

    .to_numpy()
)


col_indices = (

    positive_pairs[item_col]

    .map(item_to_idx)

    .to_numpy()
)


values = np.ones(

    len(positive_pairs),

    dtype=np.float32
)


R0_sparse = sparse.csr_matrix(

    (
        values,

        (
            row_indices,
            col_indices
        )
    ),

    shape=(
        len(users),
        len(items)
    ),

    dtype=np.float32
)


R0_sparse.sum_duplicates()

R0_sparse.data[:] = 1.0


print(
    "\n✅ Sparse R0 successfully created."
)


# ============================================================
# 21. R0 MATRIX STATISTICS
# ============================================================

n_users = (
    R0_sparse.shape[0]
)

n_items = (
    R0_sparse.shape[1]
)


total_cells = (
    n_users
    *
    n_items
)


filled_cells = (
    R0_sparse.nnz
)


density = (

    filled_cells
    /
    total_cells

    if total_cells > 0

    else 0
)


sparsity_value = (
    1
    -
    density
)


print("\n" + "=" * 70)
print("OBJECT 2 — INITIAL USER–ITEM MATRIX R0")
print("=" * 70)


print(
    f"Matrix dimensions        : "
    f"{n_users:,} × {n_items:,}"
)

print(
    f"Total possible cells     : "
    f"{total_cells:,}"
)

print(
    f"Observed positive cells  : "
    f"{filled_cells:,}"
)

print(
    f"Density                  : "
    f"{density:.8f}"
)

print(
    f"Density (%)              : "
    f"{density * 100:.6f}%"
)

print(
    f"Sparsity                 : "
    f"{sparsity_value:.8f}"
)

print(
    f"Sparsity (%)             : "
    f"{sparsity_value * 100:.6f}%"
)


# ============================================================
# 22. USER AND ITEM INTERACTION DISTRIBUTION
# ============================================================

user_interaction_counts = np.asarray(

    R0_sparse
    .sum(axis=1)

).ravel()


item_interaction_counts = np.asarray(

    R0_sparse
    .sum(axis=0)

).ravel()


# USERS

zero_interaction_users = int(

    (
        user_interaction_counts == 0
    ).sum()
)


one_interaction_users = int(

    (
        user_interaction_counts == 1
    ).sum()
)


less_than_2_users = int(

    (
        user_interaction_counts < 2
    ).sum()
)


two_or_more_users = int(

    (
        user_interaction_counts >= 2
    ).sum()
)


# ITEMS

zero_interaction_items = int(

    (
        item_interaction_counts == 0
    ).sum()
)


one_interaction_items = int(

    (
        item_interaction_counts == 1
    ).sum()
)


less_than_2_items = int(

    (
        item_interaction_counts < 2
    ).sum()
)


two_or_more_items = int(

    (
        item_interaction_counts >= 2
    ).sum()
)


print("\n" + "=" * 70)
print("INTERACTION AVAILABILITY")
print("=" * 70)


print("\nUSERS")
print("-" * 70)

print(
    f"Users with 0 positive interactions : "
    f"{zero_interaction_users:,}"
)

print(
    f"Users with exactly 1 interaction   : "
    f"{one_interaction_users:,}"
)

print(
    f"Users with <2 interactions         : "
    f"{less_than_2_users:,}"
)

print(
    f"Users with ≥2 interactions         : "
    f"{two_or_more_users:,}"
)


print("\nITEMS")
print("-" * 70)

print(
    f"Items with 0 positive interactions : "
    f"{zero_interaction_items:,}"
)

print(
    f"Items with exactly 1 interaction   : "
    f"{one_interaction_items:,}"
)

print(
    f"Items with <2 interactions         : "
    f"{less_than_2_items:,}"
)

print(
    f"Items with ≥2 interactions         : "
    f"{two_or_more_items:,}"
)


# ============================================================
# 23. COMPLETE PREPROCESSING AUDIT TABLE
# ============================================================

n_final = (
    len(df_clean)
)


n_total_removed = (

    N_RAW_PARSED
    -
    n_final
)


audit_table = pd.DataFrame({

    "Stage": [

        "Successfully parsed CSV rows",

        "Malformed CSV rows skipped",

        "Borrowed column automatically created",

        "Missing/blank Membership Number",

        "Missing/blank BIBID",

        "Missing Borrowed",

        "Invalid/non-numeric Borrowed",

        "Negative Borrowed",

        "Rows failing one or more core rules",

        "Exact duplicate rows removed",

        "Total rows removed from parsed dataset",

        "Final cleaned transaction rows",

        "Unique users",

        "Unique BIBIDs",

        "Unique user-item pairs",

        "Repeated user-item pairs",

        "Extra repeated transactions",

        "Positive user-item pairs / R0 filled cells",

        "R0 total possible cells"
    ],


    "Count": [

        N_RAW_PARSED,

        len(bad_lines_final),

        1 if borrowed_column_created else 0,

        int(
            missing_user_mask.sum()
        ),

        int(
            missing_item_mask.sum()
        ),

        int(
            missing_borrow_mask.sum()
        ),

        int(
            invalid_borrow_mask.sum()
        ),

        int(
            negative_borrow_mask.sum()
        ),

        rows_invalid_core,

        n_exact_duplicates,

        n_total_removed,

        n_final,

        n_users,

        n_items,

        len(ui_binary),

        len(repeated_pairs),

        extra_repeated_transactions,

        filled_cells,

        total_cells
    ]
})


percentage_stages = [

    "Missing/blank Membership Number",

    "Missing/blank BIBID",

    "Missing Borrowed",

    "Invalid/non-numeric Borrowed",

    "Negative Borrowed",

    "Rows failing one or more core rules",

    "Exact duplicate rows removed",

    "Total rows removed from parsed dataset",

    "Final cleaned transaction rows"
]


audit_table[
    "Percent_of_Parsed_Rows"
] = np.nan


mask_percentage = (
    audit_table["Stage"]
    .isin(
        percentage_stages
    )
)


audit_table.loc[
    mask_percentage,
    "Percent_of_Parsed_Rows"
] = (

    audit_table.loc[
        mask_percentage,
        "Count"
    ]

    /
    N_RAW_PARSED

    *
    100
)


print("\n" + "=" * 70)
print("FINAL PREPROCESSING AUDIT")
print("=" * 70)

display(audit_table)


# ============================================================
# 24. CONSISTENCY CHECKS
# ============================================================

print("\n" + "=" * 70)
print("CONSISTENCY CHECKS")
print("=" * 70)


assert (

    df_clean[user_col]
    .isna()
    .sum()

    == 0

), "Membership Number still contains missing values."


assert (

    df_clean[item_col]
    .isna()
    .sum()

    == 0

), "BIBID still contains missing values."


assert (

    df_clean[borrow_col]
    .isna()
    .sum()

    == 0

), "Borrowed still contains missing values."


assert set(

    df_clean[
        borrow_col
    ].unique()

).issubset({

    0,
    1

}), "Borrowed is not binary."


assert (

    R0_sparse.shape

    ==

    (
        df_clean[
            user_col
        ].nunique(),

        df_clean[
            item_col
        ].nunique()
    )

), "R0 dimensions do not match unique users/items."


assert (

    R0_sparse.nnz

    ==

    int(

        (
            ui_binary[
                borrow_col
            ]

            == 1

        ).sum()
    )

), "R0 positive count is inconsistent."


print("✅ No missing Membership Number")
print("✅ No missing BIBID")
print("✅ No missing Borrowed values")
print("✅ Borrowed is binary")
print("✅ R0 dimensions are correct")
print("✅ R0 positive-cell count is correct")


# ============================================================
# 25. DENSE-MATRIX MEMORY CHECK
# ============================================================

estimated_dense_gb = (

    total_cells
    *
    4

) / (

    1024 ** 3
)


print("\n" + "=" * 70)
print("DENSE MATRIX MEMORY CHECK")
print("=" * 70)


print(
    f"Estimated raw float32 memory : "
    f"{estimated_dense_gb:.2f} GB"
)


# Sparse representation remains recommended.

MAX_SPARSE_DATAFRAME_GB = 1.0


if estimated_dense_gb <= MAX_SPARSE_DATAFRAME_GB:

    R0 = pd.DataFrame.sparse.from_spmatrix(

        R0_sparse,

        index=users,

        columns=items
    )

    print(
        "✅ R0 created as a pandas sparse DataFrame."
    )


else:

    R0 = R0_sparse

    print(
        "✅ R0 retained as scipy CSR sparse matrix."
    )

    print(
        "⚠️ Dense representation was avoided "
        "to protect Colab memory."
    )


# ============================================================
# 26. SAVE PREPROCESSING OUTPUTS
# ============================================================

output_dir = (
    "/content/drive/MyDrive/Data/preprocessing_R0"
)


os.makedirs(
    output_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# OBJECT 1 — CLEANED TRANSACTION DATASET
# ------------------------------------------------------------

df_clean.to_csv(

    os.path.join(
        output_dir,
        "cleaned_transaction_dataset.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# OBJECT 2 — R0 SPARSE MATRIX
# ------------------------------------------------------------

sparse.save_npz(

    os.path.join(
        output_dir,
        "R0_sparse.npz"
    ),

    R0_sparse
)


# ------------------------------------------------------------
# USER INDEX MAPPING
# ------------------------------------------------------------

user_index_mapping.to_csv(

    os.path.join(
        output_dir,
        "R0_user_index_mapping.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# ITEM / BIBID INDEX MAPPING
# ------------------------------------------------------------

item_index_mapping.to_csv(

    os.path.join(
        output_dir,
        "R0_BIBID_index_mapping.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# UNIQUE USER–ITEM BINARY PAIRS
# ------------------------------------------------------------

ui_binary.to_csv(

    os.path.join(
        output_dir,
        "R0_unique_user_BIBID_pairs.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# PREPROCESSING AUDIT
# ------------------------------------------------------------

audit_table.to_csv(

    os.path.join(
        output_dir,
        "preprocessing_audit.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# MISSING VALUE SUMMARY
# ------------------------------------------------------------

missing_table.to_csv(

    os.path.join(
        output_dir,
        "missing_value_summary.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# DROPPED INVALID CORE RECORDS
# ------------------------------------------------------------

df_dropped_core.to_csv(

    os.path.join(
        output_dir,
        "dropped_invalid_core_records.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# DROPPED EXACT DUPLICATES
# ------------------------------------------------------------

df_dropped_duplicates.to_csv(

    os.path.join(
        output_dir,
        "dropped_exact_duplicates.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# REPEATED USER–BIBID PAIRS
# ------------------------------------------------------------

repeated_pairs.to_csv(

    os.path.join(
        output_dir,
        "repeated_user_BIBID_pairs.csv"
    ),

    index=False
)


print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)


print(f"""
Output folder:
{output_dir}

Files created:
----------------------------------------------------------------------
1. cleaned_transaction_dataset.csv
2. R0_sparse.npz
3. R0_user_index_mapping.csv
4. R0_BIBID_index_mapping.csv
5. R0_unique_user_BIBID_pairs.csv
6. preprocessing_audit.csv
7. missing_value_summary.csv
8. dropped_invalid_core_records.csv
9. dropped_exact_duplicates.csv
10. repeated_user_BIBID_pairs.csv
""")


# ============================================================
# 27. FINAL PREPROCESSING SUMMARY
# ============================================================

retention_rate = (

    len(df_clean)
    /
    N_RAW_PARSED
    *
    100

    if N_RAW_PARSED > 0

    else 0
)


removal_rate = (

    n_total_removed
    /
    N_RAW_PARSED
    *
    100

    if N_RAW_PARSED > 0

    else 0
)


borrowed_status = (

    "Automatically created; all rows assigned Borrowed = 1"

    if borrowed_column_created

    else

    "Original dataset interaction column"
)


print("\n" + "=" * 70)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 70)


print(f"""
RAW DATA
----------------------------------------------------------------------
Successfully parsed rows       : {N_RAW_PARSED:,}
Malformed CSV rows skipped     : {len(bad_lines_final):,}
Encoding                       : {successful_encoding}

CORE VARIABLES
----------------------------------------------------------------------
User identifier                : {user_col}
Item identifier                : {item_col}
Interaction variable           : {borrow_col}
Borrowed source                : {borrowed_status}

CORE DATA QUALITY
----------------------------------------------------------------------
Missing/blank Membership No.   : {int(missing_user_mask.sum()):,}
Missing/blank BIBID            : {int(missing_item_mask.sum()):,}
Missing Borrowed values        : {int(missing_borrow_mask.sum()):,}
Invalid Borrowed values        : {int(invalid_borrow_mask.sum()):,}
Negative Borrowed values       : {int(negative_borrow_mask.sum()):,}
Core-invalid rows removed      : {rows_invalid_core:,}
Exact duplicates removed       : {n_exact_duplicates:,}

OBJECT 1 — CLEANED TRANSACTION DATASET
----------------------------------------------------------------------
Final transaction rows         : {len(df_clean):,}
Rows removed                   : {n_total_removed:,}
Retention rate                 : {retention_rate:.4f}%
Removal rate                   : {removal_rate:.4f}%

USERS AND ITEMS
----------------------------------------------------------------------
Unique users                   : {n_users:,}
Unique BIBIDs                  : {n_items:,}
Unique user-item pairs         : {len(ui_binary):,}
Repeated user-item pairs       : {len(repeated_pairs):,}
Extra repeated transactions    : {extra_repeated_transactions:,}

OBJECT 2 — INITIAL MATRIX R0
----------------------------------------------------------------------
Dimensions                     : {n_users:,} × {n_items:,}
Positive observed cells        : {filled_cells:,}
Total possible cells           : {total_cells:,}
Density                        : {density:.8f}
Density (%)                    : {density * 100:.6f}%
Sparsity                       : {sparsity_value:.8f}
Sparsity (%)                   : {sparsity_value * 100:.6f}%

INTERACTION AVAILABILITY
----------------------------------------------------------------------
Users with 0 interactions      : {zero_interaction_users:,}
Users with exactly 1           : {one_interaction_users:,}
Users with <2                  : {less_than_2_users:,}
Users with ≥2                  : {two_or_more_users:,}

Items with 0 interactions      : {zero_interaction_items:,}
Items with exactly 1           : {one_interaction_items:,}
Items with <2                  : {less_than_2_items:,}
Items with ≥2                  : {two_or_more_items:,}

======================================================================
✅ PREPROCESSING COMPLETE
======================================================================
""")


# ============================================================
# 28. DISPLAY SAMPLE OF CLEANED TRANSACTION DATA
# ============================================================

print("\nSample of cleaned transaction dataset:")

display(
    df_clean.head(10)
)


# ============================================================
# 29. DISPLAY SAMPLE OF R0 USER–ITEM PAIRS
# ============================================================

print(
    "\nSample of unique user-BIBID interactions "
    "used to construct R0:"
)

display(
    ui_binary.head(10)
)


# ============================================================
# 30. OPTIONAL — SHOW FIRST 10 × 10 R0 MATRIX
# ============================================================

preview_users = min(
    10,
    R0_sparse.shape[0]
)

preview_items = min(
    10,
    R0_sparse.shape[1]
)


R0_preview = pd.DataFrame(

    R0_sparse[
        :preview_users,
        :preview_items
    ].toarray(),

    index=users[
        :preview_users
    ],

    columns=items[
        :preview_items
    ]
)


print(
    "\nFirst 10 users × first 10 BIBIDs in R0:"
)

display(
    R0_preview
)


# ============================================================
# 31. MEMORY CLEANUP
# ============================================================

gc.collect()

print(
    "\n✅ Script finished successfully."
)